# INCEpTION gold corpus - backlog from annotator comments (H)

This is a **qualitative review whose product is a list of pending work**. It is not
a measurement and must never be cited as performance evidence. The quantitative
evaluation in `E_evaluation.ipynb` deliberately excludes interpretation of comment
text, and nothing produced here feeds back into it.

Notebook H groups the verbatim inventory produced by notebook G and makes every
grouping criterion visible before applying it.


## H1. Grouping procedure and precedence

Each comment is assigned to at most one group using the following fixed precedence.
Matching is case-insensitive after whitespace collapse; regexes recognise only the
listed phrasings and close spelling variants. No category is created to absorb an
outlier.

1. **DC-01 - corpus-scope exclusions:** explicit instructions to ignore amateur,
   MASTER, GRANDMA, duplicate/previous, galaxy, or LLM-excluded measurements.
2. **CC-01 - missing representation:** an unset category plus an explicit radio,
   flux-density, photometric-colour, light-curve-reconstruction, or classification
   concept that the current row could not categorise.
3. **CC-02 - missing structured field:** no structured content field changed, but
   the text explicitly requests an evidence instrument/flux/wavelength field or a
   photometry extinction-correction field absent from the layer schema.
4. **AC-01 candidate pool - missed annotation:** any remaining row with
   `match_status == created`; structurally, the annotator added a span absent from
   the baseline. H2 describes this pool before splitting it by the created span's
   own category. Categories with at least five occurrences receive a sub-entry;
   unset and lower-frequency categories remain in `AC-01r`.
5. **AC-02 - trigger-context false positive:** a matched trigger-instrument or
   trigger-time row explicitly saying it is not a trigger, is merely an observation
   time, or should not have been highlighted.
6. **AC-03 - other false positive:** a matched row explicitly calling the annotation
   incorrect or denying the captured concept (host, redshift, spectroscopy,
   negative statement, photometry, or prompt-emission interpretation).
7. **BC-01 to BC-05 - field population:** remaining rows with structured fields in
   `changed_fields`. Photometry is split into multi-family, instrument-only,
   time/exposure-only, and measurement/calibration-only changes; evidence changes
   form one group.
8. **DC-02 - trigger-role convention:** comment-only trigger/localisation rows using
   primary/main/first-alert/multiple-instrument/earliest/best-localisation language.
9. **DC-03 - uncertain convention:** comment-only uncertainty or convention language
   about unknown/unclear values, verification, start versus mid time, filters, or
   AB/Vega.
10. **NOT ACTIONABLE:** a remaining comment-only acknowledgement or restatement with
    no action vocabulary. Anything else is **RESIDUE** and is not forced into a group.

A group with multiple annotators is labelled systematic support. A group with one
annotator is labelled as that person's practice even when it has many occurrences.


In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 360)

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "data/gcn_gold_corpus").is_dir())
INPUT = ROOT / "data/interim/gcn_gold_comments/all_annotator_comments.csv"
OUTPUT_DOCUMENT = ROOT / "docs/gcn_gold_corpus/CORRECTIONS_FROM_COMMENTS.md"
CROSSWALK_DOCUMENT = ROOT / "docs/gcn_gold_corpus/BACKLOG_CROSSWALK.md"
comments = pd.read_csv(INPUT, keep_default_na=True)
if comments.empty:
    raise ValueError(f"The comment inventory is empty: {INPUT}")
comments["annotator"] = comments["layer_source"]
comments["category"] = comments["label_or_measurement_type"].fillna("")
comments["fields"] = comments["changed_fields"].map(
    lambda value: json.loads(value) if isinstance(value, str) and value else [])
comments["normalised_text"] = comments["comment"].fillna("").map(
    lambda value: re.sub(r"\s+", " ", value.strip().casefold()))
print(f"Loaded {len(comments)} verbatim comment rows from {INPUT}")


Loaded 894 verbatim comment rows from /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_comments/all_annotator_comments.csv


In [2]:
def pattern(expression):
    return re.compile(expression, re.IGNORECASE)


SCOPE = pattern(
    r"ignore amateur|amateur ignore|amateur data|we ignore master|ignore master|"
    r"not taking into account by llm|grandma.{0,55}(ignore|no need|exclude|not include|directly in skyportal)|"
    r"gcn.{0,35}excluded.{0,25}grandma|if subject is grandma|kilonova.?catcher|grandma obs|"
    r"no need to report|no need to include|not very useful for the followup|galaxy photometry|"
    r"previous observations reported|already reported in a previous circular")
SCHEMA_CONCEPT = pattern(
    r"radio measur|radio counterpart|flux density|ujy/beam|photometric colo[u]?r|"
    r"light curve.{0,35}reconstruct|supernova flagged|comment about the classification")
SCHEMA_FIELD_EVIDENCE = pattern(r"add(ed)? instrument|missing the flux|wavelength range")
SCHEMA_FIELD_PHOTOMETRY = pattern(
    r"not corrected.{0,25}(galactic )?extinct|extinction correction|corrected from extinct")
TRIGGER_FALSE_POSITIVE = pattern(
    r"not the trigger instrument|why (is|it is) highlighted|not trigger time|not a trigger time|"
    r"start of (the )?(optical )?observations|time of the (start|observation)|1\s*sec after the trigger")
OTHER_FALSE_POSITIVE = pattern(
    r"incorrect annotation|this is incorrect|i think this is not correct|"
    r"isn.?t really giving any info on the host|there is no information about the host|"
    r"not a redshift measurement|not a redshift event|no spectroscopy was performed|"
    r"not a negative statement|not confirmation|not a photometric resul|"
    r"nothing related to the redshift|not related to the redshift|"
    r"should not be highlighted|false positive|prompt gamma-ray")
TRIGGER_GUIDE = pattern(
    r"primary trigger|first alert|main inst|swift is the main|multiple instruments|"
    r"we always take|earliest one|reference t0|better localization|usually not used|"
    r"take swift|trigger instruments showed|updated obs time/trigger|trigger context.{0,40}choose")
UNCERTAIN_GUIDE = pattern(
    r"not sure|unclear|unlcear|not clear|cannot verify|can.?t verify|how did we know|"
    r"we never know|don.?t know|do not know|unknown|certainly|usually|tradition|"
    r"verify whether|need to ask|perhaps|probably|i think|i believe|to be confirmed|"
    r"start time|startime|mid time|midtime|photometric system|ab system|vega|filter")
ACTION = pattern(
    r"\b(add|added|missing|missed|change|changed|wrong|incorrect|not|should|need|ignore|"
    r"exclude|unclear|unknown|verify|why|careful|update|updated|error|problem|must|"
    r"remove|removed|instead)\b")


GROUP_DEFINITIONS = {
    "AC-01": ("A. RULE FIXES", "Missed annotation",
              "Remaining created row: the annotator added a span absent from the baseline."),
    "AC-02": ("A. RULE FIXES", "Trigger-context false positive",
              "Matched trigger row explicitly denied as a trigger or identified as observation time."),
    "AC-03": ("A. RULE FIXES", "Other explicit false positive",
              "Matched row explicitly called incorrect or denied the captured concept."),
    "BC-01": ("B. FIELD POPULATION", "Multi-field photometry correction",
              "Photometry correction changes fields from at least two field families."),
    "BC-02": ("B. FIELD POPULATION", "Missing photometry instrument",
              "Photometry correction changes only the instrument family."),
    "BC-03": ("B. FIELD POPULATION", "Photometry time semantics",
              "Photometry correction changes only time, reference, timezone, or exposure fields."),
    "BC-04": ("B. FIELD POPULATION", "Photometry measurement fields",
              "Photometry correction changes only band, system, value/error, type, unit, target, or certainty."),
    "BC-05": ("B. FIELD POPULATION", "Evidence feature population",
              "Evidence correction changes at least one structured non-comment field."),
    "CC-01": ("C. SCHEMA GAPS", "Unrepresented scientific concept",
              "Unset category plus explicit non-optical, derived, or classification concept."),
    "CC-02": ("C. SCHEMA GAPS", "Missing structured provenance field",
              "No structured field changes despite an explicit instrument, flux, wavelength, or extinction-status need."),
    "DC-01": ("D. GUIDE GAPS", "Corpus-scope exclusions",
              "Explicit instruction to omit an amateur, MASTER, GRANDMA, duplicate, galaxy, or LLM-excluded row."),
    "DC-02": ("D. GUIDE GAPS", "Trigger-role selection convention",
              "Comment-only primary/main/first/multiple-instrument or best-localisation convention."),
    "DC-03": ("D. GUIDE GAPS", "Uncertain annotation convention",
              "Comment-only uncertainty about verification, time semantics, filters, or AB/Vega."),
    "NOT ACTIONABLE": ("NOT ACTIONABLE", "Acknowledgement or restatement",
                       "Comment-only text with no detected correction or decision vocabulary."),
}


In [3]:
TIME_FIELDS = {"obs_time_raw", "obs_time_type", "obs_time_reference",
               "timezone_raw", "exposure_time_raw"}
MEASUREMENT_FIELDS = {"photometric_system", "photometric_band", "magnitude_or_limit",
                      "magnitude_error", "limit_sigma", "measurement_type", "unit",
                      "target", "certainty"}


def assign_group(row):
    text = row["normalised_text"]
    fields = set(row["fields"]) - {"comment"}
    category = row["category"]
    if SCOPE.search(text):
        return "DC-01"
    if not category and SCHEMA_CONCEPT.search(text):
        return "CC-01"
    missing_field = (
        row["layer"] == "evidence" and SCHEMA_FIELD_EVIDENCE.search(text)
        or row["layer"] == "photometry" and SCHEMA_FIELD_PHOTOMETRY.search(text))
    if not fields and missing_field:
        return "CC-02"
    if row["match_status"] == "created":
        return "AC-01"
    if (category in {"TRIGGER_INSTRUMENT", "TRIGGER_TIME"}
            and TRIGGER_FALSE_POSITIVE.search(text)):
        return "AC-02"
    if OTHER_FALSE_POSITIVE.search(text):
        return "AC-03"
    if fields and row["layer"] == "photometry":
        families = []
        if "instrument" in fields:
            families.append("instrument")
        if fields & TIME_FIELDS:
            families.append("time")
        if fields & MEASUREMENT_FIELDS:
            families.append("measurement")
        if len(families) > 1:
            return "BC-01"
        if families == ["instrument"]:
            return "BC-02"
        if families == ["time"]:
            return "BC-03"
        if families == ["measurement"]:
            return "BC-04"
    if fields and row["layer"] == "evidence":
        return "BC-05"
    if (category in {"TRIGGER_INSTRUMENT", "TRIGGER_TIME", "LOCALIZATION"}
            and TRIGGER_GUIDE.search(text)):
        return "DC-02"
    if UNCERTAIN_GUIDE.search(text):
        return "DC-03"
    if not ACTION.search(text) or "detected correctly" in text:
        return "NOT ACTIONABLE"
    return "RESIDUE"


comments["group_id"] = comments.apply(assign_group, axis=1)
assert comments["group_id"].notna().all()
assert len(comments) == comments.groupby("group_id").size().sum()
print("Grouping is one-to-one and exhaustive including RESIDUE.")


Grouping is one-to-one and exhaustive including RESIDUE.


## H2. Decomposing the AC-01 candidate pool

### H2.1 Description before splitting

The candidate pool is first reported exactly as produced by the H1 precedence rule.
No category split is applied in the following four distributions.


In [4]:
ac01_pool = comments.loc[comments["group_id"].eq("AC-01")].copy()
ac01_pool["category_display"] = ac01_pool["category"].replace("", "<UNSET>")
ac01_by_layer = ac01_pool.groupby("layer").size().reset_index(name="occurrences")
ac01_by_category = (ac01_pool.groupby(["layer", "category_display"]).size()
                    .reset_index(name="occurrences")
                    .sort_values(["occurrences", "layer", "category_display"],
                                 ascending=[False, True, True]).reset_index(drop=True))
ac01_by_annotator = (ac01_pool.groupby("annotator").size()
                     .reset_index(name="occurrences")
                     .sort_values(["occurrences", "annotator"],
                                  ascending=[False, True]).reset_index(drop=True))
ac01_by_document = (ac01_pool.groupby("document_name").size()
                    .reset_index(name="occurrences")
                    .sort_values(["occurrences", "document_name"],
                                 ascending=[False, True]).reset_index(drop=True))
print(f"AC-01 candidate pool: {len(ac01_pool)} comments")
display(ac01_by_layer)
display(ac01_by_category)
display(ac01_by_annotator)
display(ac01_by_document)


AC-01 candidate pool: 167 comments


,layer,occurrences
0,evidence,96
1,photometry,71


,layer,category_display,occurrences
0,photometry,detection,48
1,evidence,LOCALIZATION,30
2,photometry,<UNSET>,20
3,evidence,TRIGGER_INSTRUMENT,17
4,evidence,COUNTERPART_ASSOCIATION,14
5,evidence,<UNSET>,10
6,evidence,HIGH_ENERGY_PROPERTY,9
7,evidence,REDSHIFT_EVENT,6
8,evidence,TRIGGER_TIME,4
9,photometry,upper_limit,3


,annotator,occurrences
0,Priyadarshini,57
1,Sarah,42
2,Dahlia,29
3,Yodgor,19
4,Camille,9
5,Eslam,9
6,Zhanat,2


,document_name,occurrences
0,event_GRB241030.xmi,56
1,event_GCN-251013_173943.xmi,29
2,event_GCN-251222_170549.xmi,20
3,event_EP-260623_025405.xmi,19
4,event_GRB-260708A.xmi,12
5,event_2026owq.xmi,10
6,event_GCN-260604_202037.xmi,10
7,event_2025aji.xmi,4
8,event_GCN-260614_134953.xmi,4
9,event_GRB-241025_013651.xmi,3


### H2.2 Split rule

The threshold is **five occurrences** for one populated `(layer, category)` pair.
Five is the smallest support that permits the five requested verbatim examples for
every concrete sub-entry. Unset categories and populated categories below five are
retained in the explicit heterogeneous residual `AC-01r`; they are not redistributed.

IDs follow descending measured support: `AC-01a` through `AC-01f`. The suffix `r`
is reserved for the residual and is not an alphabetical continuation.


In [5]:
AC01_THRESHOLD = 5
AC01_SPLIT_IDS = {
    ("photometry", "detection"): "AC-01a",
    ("evidence", "LOCALIZATION"): "AC-01b",
    ("evidence", "TRIGGER_INSTRUMENT"): "AC-01c",
    ("evidence", "COUNTERPART_ASSOCIATION"): "AC-01d",
    ("evidence", "HIGH_ENERGY_PROPERTY"): "AC-01e",
    ("evidence", "REDSHIFT_EVENT"): "AC-01f",
}
eligible_pairs = set(
    ac01_by_category.loc[
        ac01_by_category["category_display"].ne("<UNSET>")
        & ac01_by_category["occurrences"].ge(AC01_THRESHOLD),
        ["layer", "category_display"],
    ].itertuples(index=False, name=None))
assert eligible_pairs == set(AC01_SPLIT_IDS), (eligible_pairs, set(AC01_SPLIT_IDS))

comments["backlog_id"] = comments["group_id"]
ac01_mask = comments["group_id"].eq("AC-01")
comments.loc[ac01_mask, "backlog_id"] = comments.loc[ac01_mask].apply(
    lambda row: AC01_SPLIT_IDS.get((row["layer"], row["category"]), "AC-01r"), axis=1)

AC01_SPLIT_DEFINITIONS = {
    "AC-01a": ("A. RULE FIXES", "Missed photometry detections",
               "Created PHOTOMETRIC_MEASUREMENT rows whose measurement_type is detection."),
    "AC-01b": ("A. RULE FIXES", "Missed localizations",
               "Created evidence rows whose label is LOCALIZATION."),
    "AC-01c": ("A. RULE FIXES", "Missed trigger instruments",
               "Created evidence rows whose label is TRIGGER_INSTRUMENT."),
    "AC-01d": ("A. RULE FIXES", "Missed counterpart associations",
               "Created evidence rows whose label is COUNTERPART_ASSOCIATION."),
    "AC-01e": ("A. RULE FIXES", "Missed high-energy properties",
               "Created evidence rows whose label is HIGH_ENERGY_PROPERTY."),
    "AC-01f": ("A. RULE FIXES", "Missed event redshifts",
               "Created evidence rows whose label is REDSHIFT_EVENT."),
    "AC-01r": ("A. RULE FIXES", "Residual missed spans",
               "Created rows with an unset category or a populated category occurring fewer than five times."),
}
GROUP_DEFINITIONS.pop("AC-01")
GROUP_DEFINITIONS.update(AC01_SPLIT_DEFINITIONS)

ac01_support_rows = []
for backlog_id in ["AC-01a", "AC-01b", "AC-01c", "AC-01d",
                   "AC-01e", "AC-01f", "AC-01r"]:
    rows = comments.loc[comments["backlog_id"].eq(backlog_id)].copy()
    label_or_type = (rows["category"].iloc[0] if backlog_id != "AC-01r"
                     else "<UNSET or category count < 5>")
    ac01_support_rows.append({
        "ID": backlog_id, "layer": ", ".join(sorted(rows["layer"].unique())),
        "label_or_measurement_type": label_or_type,
        "occurrences": len(rows), "distinct_annotators": rows["annotator"].nunique(),
        "distinct_documents": rows["document_name"].nunique(),
    })
ac01_subentry_table = pd.DataFrame(ac01_support_rows)
display(ac01_subentry_table)

residual_members = (comments.loc[comments["backlog_id"].eq("AC-01r")]
                    .assign(category_display=lambda frame: frame["category"].replace("", "<UNSET>"))
                    .groupby(["layer", "category_display"]).size()
                    .reset_index(name="occurrences")
                    .sort_values(["occurrences", "layer", "category_display"],
                                 ascending=[False, True, True]))
print("\nAC-01r membership:")
display(residual_members)

for backlog_id in ac01_subentry_table["ID"]:
    rows = comments.loc[comments["backlog_id"].eq(backlog_id)].copy()
    examples = (rows.sort_values(
        ["normalised_text", "document_name", "begin"], kind="stable")
        .drop_duplicates(["normalised_text", "covered_text"]).head(5))
    assert len(examples) == 5, (backlog_id, len(examples))
    print(f"\n{backlog_id} - {GROUP_DEFINITIONS[backlog_id][1]}")
    for example in examples.itertuples(index=False):
        span_text = str(example.covered_text).replace("\n", "\\n")
        print(f"  COMMENT: {example.comment}")
        print(f"  SPAN: {span_text}")


,ID,layer,label_or_measurement_type,occurrences,distinct_annotators,distinct_documents
0,AC-01a,photometry,detection,48,6,9
1,AC-01b,evidence,LOCALIZATION,30,5,9
2,AC-01c,evidence,TRIGGER_INSTRUMENT,17,3,6
3,AC-01d,evidence,COUNTERPART_ASSOCIATION,14,4,4
4,AC-01e,evidence,HIGH_ENERGY_PROPERTY,9,3,4
5,AC-01f,evidence,REDSHIFT_EVENT,6,4,5
6,AC-01r,"evidence, photometry",<UNSET or category count < 5>,43,5,7



AC-01r membership:


,layer,category_display,occurrences
7,photometry,<UNSET>,20
0,evidence,<UNSET>,10
6,evidence,TRIGGER_TIME,4
8,photometry,upper_limit,3
3,evidence,LIGHTCURVE_EVOLUTION,2
1,evidence,CLASSIFICATION_INTERPRETATION,1
2,evidence,EVENT_IDENTITY,1
4,evidence,REDSHIFT_CONTEXT,1
5,evidence,SPECTROSCOPY,1



AC-01a - Missed photometry detections
  COMMENT: Added a photometric measurement.
  SPAN: 2025-12-23  18:56:19    ~25.84   I     200s*8     19.55 +/-0.02
  COMMENT: Added by annotator
  SPAN: g = 19.56 +/- 0.02
  COMMENT: Added the photometry point
  SPAN: 2025-12-22  20:28:35    ~3.38   R     300s*5     18.39 +/-0.01
  COMMENT: Added the photometry tag here.
  SPAN: 15.42 with a 1-sigma error of about  0.14.
  COMMENT: Added this annotation for the photometric value; circular doesn't give photometric system and only mentions using PanSTARRS
  SPAN: 2025-12-22   18:56:55 UT   1.85           30x60s           CR        18.70     +/- 0.09

AC-01b - Missed localizations
  COMMENT: Added a localization tag
  SPAN: 90% confidence level (C.L.) radius of 2.75 arcmin
  COMMENT: Added a localization tag for the BAT location
  SPAN: RA(J2000)  =	22:52:33.57 = 343.13987\n  DEC(J2000) = +80:26:59.9  =  80.44996
  COMMENT: Added a localization tag for the XRT couterpart
  SPAN: RA(J2000)  = 22h 52m

## H3. Group support and verbatim evidence

Counts refer to comments, not independent scientific errors. Five distinct verbatim
texts are printed where available; smaller groups print every distinct text.


In [6]:
grouped = comments.loc[comments["backlog_id"].ne("RESIDUE")].copy()
group_table_rows = []
for backlog_id, rows in grouped.groupby("backlog_id", sort=False):
    category, title, criterion = GROUP_DEFINITIONS[backlog_id]
    annotators = sorted(rows["annotator"].unique())
    group_table_rows.append({
        "ID": backlog_id, "category": category, "group": title,
        "criterion": criterion, "comments": len(rows),
        "distinct_annotators": len(annotators),
        "distinct_documents": rows["document_name"].nunique(),
        "support": ("systematic (multiple annotators)" if len(annotators) > 1
                    else f"single-annotator practice ({annotators[0]})"),
    })
group_table = pd.DataFrame(group_table_rows).sort_values(
    ["category", "ID"]).reset_index(drop=True)
display(group_table)

for backlog_id in group_table["ID"]:
    rows = comments.loc[comments["backlog_id"].eq(backlog_id)].copy()
    examples = (rows.groupby("normalised_text", as_index=False)
                .agg(count=("normalised_text", "size"), text=("comment", "first"))
                .sort_values(["count", "normalised_text"], ascending=[False, True])
                .head(5))
    print(f"\n{backlog_id} - {GROUP_DEFINITIONS[backlog_id][1]}")
    for example in examples.itertuples(index=False):
        print(f"  [{example.count}] {example.text}")


,ID,category,group,criterion,comments,distinct_annotators,distinct_documents,support
0,AC-01a,A. RULE FIXES,Missed photometry detections,Created PHOTOMETRIC_MEASUREMENT rows whose measurement_type is detection.,48,6,9,systematic (multiple annotators)
1,AC-01b,A. RULE FIXES,Missed localizations,Created evidence rows whose label is LOCALIZATION.,30,5,9,systematic (multiple annotators)
2,AC-01c,A. RULE FIXES,Missed trigger instruments,Created evidence rows whose label is TRIGGER_INSTRUMENT.,17,3,6,systematic (multiple annotators)
3,AC-01d,A. RULE FIXES,Missed counterpart associations,Created evidence rows whose label is COUNTERPART_ASSOCIATION.,14,4,4,systematic (multiple annotators)
4,AC-01e,A. RULE FIXES,Missed high-energy properties,Created evidence rows whose label is HIGH_ENERGY_PROPERTY.,9,3,4,systematic (multiple annotators)
5,AC-01f,A. RULE FIXES,Missed event redshifts,Created evidence rows whose label is REDSHIFT_EVENT.,6,4,5,systematic (multiple annotators)
6,AC-01r,A. RULE FIXES,Residual missed spans,Created rows with an unset category or a populated category occurring fewer than five times.,43,5,7,systematic (multiple annotators)
7,AC-02,A. RULE FIXES,Trigger-context false positive,Matched trigger row explicitly denied as a trigger or identified as observation time.,9,3,3,systematic (multiple annotators)
8,AC-03,A. RULE FIXES,Other explicit false positive,Matched row explicitly called incorrect or denied the captured concept.,9,5,5,systematic (multiple annotators)
9,BC-01,B. FIELD POPULATION,Multi-field photometry correction,Photometry correction changes fields from at least two field families.,96,6,7,systematic (multiple annotators)



AC-01a - Missed photometry detections
  [4] Added this photometry tag.
  [4] I created this photometric measurement as this one was missing. I had trouble choosing the span as the tc-t0(s) and t_exp are relevant to the g' filter but the i' mag and its error is not relevant. I entered the photometric system to be AB since its Sloan filter and photometric band to be g' as this is the filter corresponding to the magnitude value here. I also added the error. I also added the obse_time_raw, the obs_time_type and obs_time_reference
  [4] obs_start_time !
  [4] this photometry point was not at all highlighted i added all the info
  [3] Added this photometry tag. AB since ATLAS is AB photometric system.

AC-01b - Missed localizations
  [2] Added localization uncertainty (90% confidence radius)
  [2] Added the localization tag
  [1] Added a localization tag
  [1] Added a localization tag for the BAT location
  [1] Added a localization tag for the XRT couterpart

AC-01c - Missed trigger instrum


BC-01 - Multi-field photometry correction
  [8] changed absolute_time to unknown. its unclear what the mjd is. added instrument.
  [6] observation time; photometric band; and system were not detcted although its clear in the text. I added the time, band, system, instrument.
  [5] AB system, time and system present in the circular
  [4] Changed tentative to confirmed, changed the obs_time_reference from absolute to observation_start, added the exposure and instrument name
  [4] changed to observation_start and added instrument.

BC-02 - Missing photometry instrument
  [14] Added instrument name
  [9] how did we know this is vega? Added inst as UVOT. time is relative to trigger but should be indicated that its T-start.
  [7] photometric system is unknown. Added instrument name.
  [6] Added the instrument
  [6] Added the instrument name

BC-03 - Photometry time semantics
  [17] photometric system is unknown. changed to observation_start
  [8] changed absolute_time to observation_mid
  [8


DC-02 - Trigger-role selection convention
  [15] Primary trigger instrument - first alert
  [8] primary trigger instrument
  [7] swift is the main inst
  [2] Trigger instruments showed correctly and in the right order: Fermi/GBM, SWIFT/BAT, SWIFT/XRT
  [1] first localization usually not used as we take the skylocalization, we search more candidates within the circle

DC-03 - Uncertain annotation convention
  [8] photometric system is unknown. certainly VEGA.
  [7] photometric system is unknown / start time and not mid time
  [5] start time
  [4] need to be taken the midtime and not startime
  [4] photometric system is unknown. Certainly AB.

NOT ACTIONABLE - Acknowledgement or restatement
  [40] AB
  [11] Fermi/GBM
  [4] Fermi/LAT
  [3] AB / V or clear band
  [3] T-T0 in h given, mention Gaia Bp (eq Blue), AB (written below)


## H4. Unclassified residue

Residue is reported rather than expanded into an after-the-fact category.


In [7]:
residue = comments.loc[comments["group_id"].eq("RESIDUE")].copy()
print(f"Residue: {len(residue)} of {len(comments)} comments "
      f"({100 * len(residue) / len(comments):.2f}%).")
residue_examples = (residue.sort_values(
    ["document_name", "annotator", "begin"], kind="stable")[[
        "document_name", "annotator", "layer", "label_or_measurement_type",
        "begin", "end", "covered_text", "comment",
    ]].head(10))
display(residue_examples)


Residue: 7 of 894 comments (0.78%).


,document_name,annotator,layer,label_or_measurement_type,begin,end,covered_text,comment
41,event_2026owq.xmi,Sarah,evidence,COUNTERPART_ASSOCIATION,1188,1217,optical counterpart candidate,GOTO can make error so it is great to say it is a candidate
154,event_EP-260623_025405.xmi,Dahlia,evidence,LOCALIZATION,856,894,"R.A. = 328.2725 deg, DEC = 12.7946 deg","This is xray counterpart localization, not to be confused with the 3 arcmin localization by WXT"
218,event_GCN-251013_173943.xmi,Sarah,evidence,LOCALIZATION,30318,30369,RA (J2000) = 23:03:20.56\nDec (J2000) = -00:12:37.21,radio counterpart must be coherent with Optical
441,event_GCN-251222_170549.xmi,Priyadarshini,evidence,REDSHIFT_EVENT,44890,44899,z = 3.171,Attributed to the GRB based on sentence context and cited GCN 43204; not a contextual redshift.
443,event_GCN-251222_170549.xmi,Priyadarshini,evidence,REDSHIFT_EVENT,49548,49557,z = 3.171,Attributed to the GRB based on sentence context and cited GCN 43204; not a contextual redshift.
486,event_GCN-260604_202037.xmi,Sarah,evidence,LIGHTCURVE_EVOLUTION,39118,39129,brightening,should be good to associated with 2.86 days
490,event_GCN-260604_202037.xmi,Sarah,evidence,LIGHTCURVE_EVOLUTION,45579,45592,rebrightening,add aproximately 30h post T0


## H5. Concrete backlog entries

The document is generated from the grouped rows below. `Likely cause` remains
explicitly uncertain where the comments do not establish implementation causality.
Confidence is high only for groups supported by multiple annotators and documents.


In [8]:
BACKLOG = {
    "AC-01a": {
        "symptom": "Valid photometric detections are absent from the baseline and created manually.",
        "cause": "The missed spans include compact rows and prose measurements; the comments do not establish one parser defect for all 48.",
        "fix": "Stratify these spans by syntax, add real-snippet tests per recurrent format, and extend only the matching row or prose rule.",
        "effort": "high",
    },
    "AC-01b": {
        "symptom": "Coordinates and localization uncertainties are absent from the baseline.",
        "cause": "Missed phrasings include multiline RA/Dec blocks and standalone confidence-radius expressions; no single cause covers every span.",
        "fix": "Add localization tests for the quoted coordinate and uncertainty forms, retaining whole coordinate blocks where present.",
        "effort": "medium",
    },
    "AC-01c": {
        "symptom": "Explicit trigger-instrument mentions are absent from the baseline.",
        "cause": "Quoted spans include `Fermi GBM`, `Swift/BAT`, `SVOM`, and `SVOM/ECLAIRs`; the missed governing phrasings vary.",
        "fix": "Audit each surrounding trigger clause and add instrument-specific regression cases without accepting follow-up-only mentions.",
        "effort": "medium",
    },
    "AC-01d": {
        "symptom": "Counterpart or afterglow associations are absent from the baseline.",
        "cause": "Missed spans include `point source`, `optical afterglow`, `near-infrared transient`, and `possibly associated` phrasing.",
        "fix": "Extend counterpart-association phrasing with the quoted spans and preserve candidate/tentative context.",
        "effort": "medium",
    },
    "AC-01e": {
        "symptom": "High-energy quantities or evolution statements are absent from the baseline.",
        "cause": "Missed spans include photon flux, highest-energy photon, column density, isotropic luminosity, and a post-break decay index.",
        "fix": "Add subtype-specific tests for each quoted quantity; do not replace them with one broad numeric high-energy fallback.",
        "effort": "high",
    },
    "AC-01f": {
        "symptom": "Annotators created REDSHIFT_EVENT spans absent from the baseline.",
        "cause": "The six spans mix explicit, tentative, and context-dependent redshift phrasing; one even requires checking whether the span is a colour expression.",
        "fix": "Review the six cases individually, then add referent-aware tests only for confirmed event-redshift phrasings.",
        "effort": "medium",
    },
    "AC-01r": {
        "symptom": "Forty-three created rows remain heterogeneous or below the five-occurrence threshold.",
        "cause": "Not inferable as one cause: the residual combines unset categories and several low-frequency labels/types.",
        "fix": "Triage by the residual membership table before promoting any pattern to a standalone rule or schema task.",
        "effort": "high", "confidence": "low",
    },
    "AC-02": {
        "symptom": "Trigger labels include follow-up instruments or observation times that are not triggers.",
        "cause": "Trigger rules do not consistently enforce governing trigger context.",
        "fix": "Add governing-clause and instrument-role gates, preserving explicit trigger detections and times.",
        "effort": "medium",
    },
    "AC-03": {
        "symptom": "Matched spans are explicitly denied as the concept assigned by the extractor.",
        "cause": "The cases span several labels; one common implementation cause is not clear.",
        "fix": "Audit each occurrence by label and add narrow negative tests before changing any rule.",
        "effort": "high",
    },
    "BC-01": {
        "symptom": "One photometry row requires corrections across multiple field families.",
        "cause": "Row/clause parsing does not propagate all locally governed fields into one measurement.",
        "fix": "Populate instrument, timing, band/system, and numeric fields from the same parsed row or governing clause.",
        "effort": "high",
    },
    "BC-02": {
        "symptom": "The photometric span is retained but its instrument is added or changed.",
        "cause": "Instrument attribution is absent or too conservative for explicit local evidence.",
        "fix": "Use explicit table columns, captions, and governing clauses; never borrow a distant instrument mention.",
        "effort": "medium",
    },
    "BC-03": {
        "symptom": "Observation time, reference, type, timezone, or exposure fields are corrected.",
        "cause": "Start, mid-exposure, absolute, and trigger-relative time semantics are not consistently distinguished.",
        "fix": "Parse the local time expression and its governing header/clause into the existing time fields with tests.",
        "effort": "high",
    },
    "BC-04": {
        "symptom": "Band, photometric system, value/error, type, unit, target, or certainty is corrected.",
        "cause": "Calibration and measurement metadata are not fully recovered from local text/table structure.",
        "fix": "Add explicit local parsing and vocabulary tests; preserve unknown rather than infer an unsupported system.",
        "effort": "medium",
    },
    "BC-05": {
        "symptom": "Evidence spans are retained but label, target, certainty, value, or unit changes.",
        "cause": "Label-specific feature normalisation does not match all annotator conventions.",
        "fix": "Split by changed field and label, then add controlled-vocabulary and context tests per extractor.",
        "effort": "medium",
    },
    "CC-01": {
        "symptom": "Comments name scientific concepts while leaving the row category unset.",
        "cause": "The comments indicate representational pressure, but the appropriate existing or new layer is not clear.",
        "fix": "Make a schema decision for each concept before adding extraction rules; use a new layer only where no existing layer fits.",
        "effort": "high",
    },
    "CC-02": {
        "symptom": "Comments request provenance/detail that has no structured field change.",
        "cause": "Evidence lacks dedicated instrument/flux/wavelength provenance and photometry lacks extinction-correction status.",
        "fix": "Decide whether these belong in new optional fields or a separate relation/provenance representation.",
        "effort": "high",
    },
    "DC-01": {
        "symptom": "Annotators apply undocumented inclusion exclusions to otherwise valid measurements.",
        "cause": "The guide does not define corpus scope for amateur, network-duplicate, galaxy, or previously reported data.",
        "fix": "Define a machine-readable inclusion policy and annotate scope separately from extraction correctness.",
        "effort": "medium",
    },
    "DC-02": {
        "symptom": "Comments disagree or elaborate on which trigger instrument/time/localisation is primary.",
        "cause": "The guide does not define role and precedence when several instruments or localisations are reported.",
        "fix": "Document per-circular capture versus event-level primary selection and add a role field only if both must coexist.",
        "effort": "medium",
    },
    "DC-03": {
        "symptom": "Annotators record unresolved conventions for time semantics, filters, systems, or uncertain values.",
        "cause": "The guide lacks decisive examples for unknown values and start/mid/AB/Vega conventions.",
        "fix": "Add controlled examples and an explicit unknown policy; do not require unsupported inference.",
        "effort": "low",
    },
}


def markdown_escape(value):
    return str(value).replace("\n", "\\n").replace("|", "\\|")


BACKLOG_ORDER = [
    "AC-01a", "AC-01b", "AC-01c", "AC-01d", "AC-01e", "AC-01f", "AC-01r",
    "AC-02", "AC-03", "BC-01", "BC-02", "BC-03", "BC-04", "BC-05",
    "CC-01", "CC-02", "DC-01", "DC-02", "DC-03",
]
category_order = ["A. RULE FIXES", "B. FIELD POPULATION", "C. SCHEMA GAPS", "D. GUIDE GAPS"]
document_lines = [
    "# Corrections From Annotator Comments", "",
    "> This is a qualitative review whose product is a list of pending work. "
    "It is not a measurement and must never be cited as performance evidence.", "",
    "Generated by `notebooks/gcn_gold/H_comments_backlog.ipynb` from the verbatim "
    "inventory produced by notebook G. Comment counts are not independent error counts.", "",
    "The former catch-all AC-01 is split at a threshold of five occurrences per populated "
    "layer/category pair; lower-frequency and unset rows remain in AC-01r.", "",
]
summary_rows = []
for category_name in category_order:
    document_lines.extend([f"## {category_name}", ""])
    category_ids = [backlog_id for backlog_id in BACKLOG_ORDER
                    if GROUP_DEFINITIONS[backlog_id][0] == category_name]
    for backlog_id in category_ids:
        rows = comments.loc[comments["backlog_id"].eq(backlog_id)].copy()
        title = GROUP_DEFINITIONS[backlog_id][1]
        metadata = BACKLOG[backlog_id]
        annotators = sorted(rows["annotator"].unique())
        documents = rows["document_name"].nunique()
        confidence = metadata.get(
            "confidence", "high" if len(annotators) > 1 and documents > 1 else "low")
        labels = (rows.assign(category_display=rows["category"].replace("", "<UNSET>"))
                  .groupby("layer")["category_display"]
                  .apply(lambda values: ", ".join(sorted(set(values)))))
        layer_label = "; ".join(f"{layer}: {values}" for layer, values in labels.items())
        evidence_rows = (rows.sort_values(
            ["normalised_text", "document_name", "begin"], kind="stable")
            .drop_duplicates(["normalised_text", "covered_text"]).head(5))
        document_lines.extend([
            f"### [{backlog_id}] {title}",
            f"- **Reported by:** {', '.join(annotators)}",
            f"- **Layer / label:** {layer_label}",
            f"- **Symptom:** {metadata['symptom']}",
            f"- **Evidence:** {len(rows)} comments; {len(annotators)} distinct annotators; "
            f"{documents} distinct documents.",
        ])
        for evidence in evidence_rows.itertuples(index=False):
            document_lines.append(
                f"  - Comment: \"{markdown_escape(evidence.comment)}\"; "
                f"span: \"{markdown_escape(evidence.covered_text)}\"")
        document_lines.extend([
            f"- **Likely cause:** {metadata['cause']}",
            f"- **Suggested fix:** {metadata['fix']}",
            f"- **Effort:** {metadata['effort']}",
            f"- **Confidence:** {confidence}", "",
        ])
        summary_rows.append({
            "ID": backlog_id, "category": category_name,
            "layer": ", ".join(sorted(rows["layer"].unique())),
            "occurrences": len(rows), "distinct_annotators": len(annotators),
            "effort": metadata["effort"], "confidence": confidence,
        })

document_lines.extend([
    "## Unclassified And Non-Actionable Comments", "",
    f"- NOT ACTIONABLE: {int(comments['backlog_id'].eq('NOT ACTIONABLE').sum())} comments.",
    f"- RESIDUE: {int(comments['backlog_id'].eq('RESIDUE').sum())} comments; no category was forced.", "",
])
OUTPUT_DOCUMENT.write_text("\n".join(document_lines), encoding="utf-8")
backlog_summary = pd.DataFrame(summary_rows)
print(f"Wrote {OUTPUT_DOCUMENT} ({len(document_lines)} generated lines)")


Wrote /home/meneses/project_astronomical/MAFORAI/docs/gcn_gold_corpus/CORRECTIONS_FROM_COMMENTS.md (306 generated lines)


## H6. Backlog summary

Occurrences are comment rows. Confidence follows the support rule stated above.


In [9]:
display(backlog_summary)


,ID,category,layer,occurrences,distinct_annotators,effort,confidence
0,AC-01a,A. RULE FIXES,photometry,48,6,high,high
1,AC-01b,A. RULE FIXES,evidence,30,5,medium,high
2,AC-01c,A. RULE FIXES,evidence,17,3,medium,high
3,AC-01d,A. RULE FIXES,evidence,14,4,medium,high
4,AC-01e,A. RULE FIXES,evidence,9,3,high,high
5,AC-01f,A. RULE FIXES,evidence,6,4,medium,high
6,AC-01r,A. RULE FIXES,"evidence, photometry",43,5,high,low
7,AC-02,A. RULE FIXES,evidence,9,3,medium,high
8,AC-03,A. RULE FIXES,evidence,9,5,high,high
9,BC-01,B. FIELD POPULATION,photometry,96,6,high,high


### H6.1 Generated document

`docs/gcn_gold_corpus/CORRECTIONS_FROM_COMMENTS.md` is generated by this notebook,
not maintained by hand. Its opening explicitly separates qualitative pending work
from quantitative evaluation.


In [10]:
generated_text = OUTPUT_DOCUMENT.read_text(encoding="utf-8")
print(f"Generated document lines: {len(generated_text.splitlines())}")
print("\n".join(generated_text.splitlines()[:8]))


Generated document lines: 305
# Corrections From Annotator Comments

> This is a qualitative review whose product is a list of pending work. It is not a measurement and must never be cited as performance evidence.

Generated by `notebooks/gcn_gold/H_comments_backlog.ipynb` from the verbatim inventory produced by notebook G. Comment counts are not independent error counts.

The former catch-all AC-01 is split at a threshold of five occurrences per populated layer/category pair; lower-frequency and unset rows remain in AC-01r.



## H7. Crosswalk with the Slack-derived backlog

The Slack document is parsed from disk and compared with the generated comment
entries by symptom rather than identifier. `yes` means symptom-level corroboration;
`partial` means the evidence overlaps but one source is broader or differently
framed; `no` means no counterpart was identified. This remains qualitative review.


In [11]:
slack_backlog = ROOT / "docs/gcn_gold_corpus/CORRECTIONS_FROM_ANNOTATORS.md"
if not slack_backlog.exists():
    raise FileNotFoundError(slack_backlog)
slack_text = slack_backlog.read_text(encoding="utf-8")
slack_entries = re.findall(r"^### \[([A-D]-\d{2})\] (.+)$", slack_text, flags=re.MULTILINE)
assert len(slack_entries) == 37, len(slack_entries)

CROSSWALK_SPECS = {
    "A-01": ("AC-01a", "yes", "Both sources report valid photometric detections absent from the baseline."),
    "A-02": ("AC-01a", "partial", "The 48 missed detections include compact/table spans, but AC-01a is not syntax-specific."),
    "A-03": ("AC-01r", "partial", "AC-01r contains two low-support LIGHTCURVE_EVOLUTION creations, not a dedicated decline-from-X-to-Y group."),
    "A-04": ("AC-01d, CC-01, CC-02", "yes", "Comments independently record missed radio counterparts, radio measurements, and missing radio flux."),
    "A-05": ("AC-01b", "yes", "Thirty created localization spans include BAT/XRT/UVOT coordinates and confidence radii."),
    "A-06": ("AC-01e, AC-03, BC-05", "partial", "Comments show missed high-energy quantities and modality relabels, but not every Slack instrument-binding case."),
    "A-07": ("BC-05", "yes", "A structured correction explicitly says event redshift rather than REDSHIFT_CONTEXT."),
    "A-08": ("CC-01", "yes", "A comment explicitly identifies a colour expression as photometric rather than redshift."),
    "A-09": ("CC-01", "yes", "A comment explicitly identifies the SN1998bw-like statement as classification information."),
    "A-10": ("AC-02", "yes", "Five TRIGGER_INSTRUMENT comments explicitly deny trigger role; the comments support and extend the Slack case set."),
    "A-11": ("AC-02", "yes", "Four TRIGGER_TIME comments identify observation/start/offset times rather than triggers."),
    "A-12": ("AC-03, DC-03", "yes", "A comment states that the conditional absorption phrase is not a NEGATIVE_STATEMENT."),
    "B-01": ("BC-01, BC-03, DC-03", "yes", "Both sources repeatedly distinguish absolute, start, mid, and trigger-relative time semantics."),
    "B-02": ("BC-03, DC-03", "yes", "Comments explicitly request mid-time instead of start-time in recurrent rows."),
    "B-03": ("BC-03, BC-05", "partial", "Comments cover missing date context and time-field changes, but combine photometry and trigger-time cases."),
    "B-04": ("BC-05", "yes", "Evidence comments repeatedly complete trigger times with dates recovered from circular context."),
    "B-05": ("BC-01, BC-04, DC-03", "yes", "AB/Vega population and uncertainty conventions recur throughout the comments."),
    "B-06": ("BC-01, BC-02", "yes", "Instrument additions are a recurrent structured photometry correction."),
    "B-07": ("BC-01, BC-04", "yes", "Photometry field corrections include magnitude-error population."),
    "B-08": ("BC-01, BC-04", "yes", "Photometric-band population is represented among single- and multi-field corrections."),
    "C-01": ("DC-02", "partial", "Comments strongly request a primary trigger convention; they do not by themselves decide field versus guide design."),
    "C-02": ("AC-01e, CC-01", "partial", "Comments show missed high-energy quantities and unrepresented non-optical measurements, but not a complete X-ray schema."),
    "C-03": ("CC-02", "yes", "Comments repeatedly request explicit Galactic-extinction correction status absent from structured fields."),
    "C-04": ("", "no", "No comment-derived backlog entry isolates a structured epoch for LIGHTCURVE_EVOLUTION."),
    "C-05": ("AC-01r", "partial", "The residual contains one T90-accuracy comment, but not a systematic T90-by-energy-band representation task."),
    "C-06": ("", "no", "No comment-derived backlog entry represents relations among successive localizations."),
    "D-01": ("DC-02", "yes", "Both sources record ambiguity in selecting the primary/best trigger time and source."),
    "D-02": ("BC-05, DC-03", "partial", "Comments cover certainty/value changes and uncertain conventions, but not the full span-boundary rubric."),
    "D-03": ("", "no", "No comment-derived backlog entry concerns review_priority."),
    "D-04": ("", "no", "No comment-derived backlog entry concerns EVENT_SUMMARY workflow."),
    "D-05": ("BC-04, DC-03", "yes", "Comments repeatedly expose explicit and inferred AB/Vega conventions."),
    "D-06": ("BC-04, DC-03", "partial", "Comments discuss filter equivalences and band population, but do not establish a complete normalization policy."),
    "D-07": ("AC-01a", "partial", "Missing detections corroborate the annotation-unit problem, but AC-01a does not isolate multi-value sentences."),
    "D-08": ("DC-01", "yes", "DC-01 directly reproduces MASTER, amateur, GRANDMA, duplicate, and LLM scope exclusions."),
    "D-09": ("DC-01", "yes", "DC-01 includes explicit comments excluding galaxy photometry from transient measurements."),
    "D-10": ("CC-01", "yes", "A comment explicitly identifies a supernova interpretation needing classification representation."),
    "D-11": ("BC-01, BC-04", "partial", "Comments support magnitude-error population, but do not fully encode the span-editing workflow."),
}
assert set(CROSSWALK_SPECS) == {entry_id for entry_id, _ in slack_entries}

crosswalk_rows = []
mapped_comment_ids = set()
for slack_id, _title in slack_entries:
    comment_ids, status, relation = CROSSWALK_SPECS[slack_id]
    ids = [value.strip() for value in comment_ids.split(",") if value.strip()]
    mapped_comment_ids.update(ids)
    crosswalk_rows.append({
        "Slack ID": slack_id, "comment-derived ID": comment_ids,
        "same symptom?": status, "relationship": relation,
    })

comment_entry_ids = set(backlog_summary["ID"])
comment_only_relations = {
    "AC-01c": "Comments report missing trigger-instrument spans; Slack A-10 instead concerns false-positive follow-up instruments.",
    "AC-01f": "Comments report entirely missed REDSHIFT_EVENT spans; Slack A-07 concerns an existing span with the wrong redshift label.",
}
assert comment_entry_ids - mapped_comment_ids == set(comment_only_relations)
for comment_id, relation in comment_only_relations.items():
    crosswalk_rows.append({
        "Slack ID": "", "comment-derived ID": comment_id,
        "same symptom?": "no", "relationship": relation,
    })
crosswalk = pd.DataFrame(crosswalk_rows)
display(crosswalk)

confirmed_both = crosswalk.loc[crosswalk["same symptom?"].eq("yes")]
partial_overlap = crosswalk.loc[crosswalk["same symptom?"].eq("partial")]
slack_only = crosswalk.loc[
    crosswalk["Slack ID"].ne("") & crosswalk["comment-derived ID"].eq("")]
comments_only = crosswalk.loc[
    crosswalk["Slack ID"].eq("") & crosswalk["comment-derived ID"].ne("")]

def id_pairs(frame):
    return "; ".join(
        f"{row['Slack ID']} <-> {row['comment-derived ID']}" for _, row in frame.iterrows())


print("Symptoms confirmed by BOTH sources:")
print(id_pairs(confirmed_both))
print("\nSymptoms only in Slack:")
print(", ".join(slack_only["Slack ID"]) or "None")
print("\nSymptoms only in comments:")
print(", ".join(comments_only["comment-derived ID"]) or "None")
print("\nPartial correspondences (not counted as confirmed by both):")
print(id_pairs(partial_overlap))

dc01_rows = comments.loc[comments["backlog_id"].eq("DC-01")]
trigger_rows = comments.loc[
    comments["layer"].eq("evidence") & comments["category"].eq("TRIGGER_INSTRUMENT")]
trigger_breakdown = (trigger_rows.groupby("backlog_id").size()
                     .reset_index(name="comments")
                     .sort_values(["comments", "backlog_id"], ascending=[False, True]))
print(f"\nDC-01 versus D-08: SUPPORTS AND EXTENDS. DC-01 has {len(dc01_rows)} comments, "
      f"{dc01_rows['annotator'].nunique()} annotators, and {dc01_rows['document_name'].nunique()} documents. "
      "It directly supports MASTER/amateur/GRANDMA exclusions and extends them to duplicate, galaxy, and LLM-excluded rows.")
print(f"TRIGGER_INSTRUMENT versus A-10: SUPPORTS AND EXTENDS, DOES NOT CONTRADICT. "
      f"TRIGGER_INSTRUMENT is the most-commented evidence label with {len(trigger_rows)} comments; "
      f"AC-02 contains {int((trigger_rows['backlog_id'] == 'AC-02').sum())} explicit false-positive rows. "
      "The remaining comments extend the issue to missed trigger spans, field corrections, and primary-role conventions.")
display(trigger_breakdown)

crosswalk_lines = [
    "# Backlog Crosswalk", "",
    "This document maps two independently produced backlogs: one derived from private Slack "
    "messages and one derived from in-corpus annotator comments. Symptom-level overlap is "
    "corroboration, not duplication; counts remain qualitative and are not performance metrics.", "",
    "## Correspondence", "",
    "| Slack ID | Comment-derived ID | Same symptom? | Relationship |",
    "|---|---|---|---|",
]
for row in crosswalk.itertuples(index=False):
    crosswalk_lines.append(
        f"| {markdown_escape(row[0])} | {markdown_escape(row[1])} | "
        f"{markdown_escape(row[2])} | {markdown_escape(row[3])} |")
crosswalk_lines.extend([
    "", "## Symptoms Confirmed By Both Sources", "",
    id_pairs(confirmed_both), "", "## Symptoms Only In Slack", "",
    ", ".join(slack_only["Slack ID"]) or "None", "",
    "## Symptoms Only In Comments", "",
    ", ".join(comments_only["comment-derived ID"]) or "None", "",
    "## Partial Correspondences", "", id_pairs(partial_overlap), "",
    "## Required Spot Checks", "",
    f"- **DC-01 versus D-08:** supports and extends. DC-01 contains {len(dc01_rows)} comments "
    f"from {dc01_rows['annotator'].nunique()} annotators across {dc01_rows['document_name'].nunique()} documents.",
    f"- **TRIGGER_INSTRUMENT versus A-10:** supports and extends, without contradiction. "
    f"There are {len(trigger_rows)} TRIGGER_INSTRUMENT comments, including "
    f"{int((trigger_rows['backlog_id'] == 'AC-02').sum())} explicit false-positive rows.", "",
])
CROSSWALK_DOCUMENT.write_text("\n".join(crosswalk_lines), encoding="utf-8")
print(f"\nWrote {CROSSWALK_DOCUMENT} ({len(crosswalk_lines)} generated lines)")
print("\nThis notebook remains a qualitative backlog derivation, not a measurement.")


,Slack ID,comment-derived ID,same symptom?,relationship
0,A-01,AC-01a,yes,Both sources report valid photometric detections absent from the baseline.
1,A-02,AC-01a,partial,"The 48 missed detections include compact/table spans, but AC-01a is not syntax-specific."
2,A-03,AC-01r,partial,"AC-01r contains two low-support LIGHTCURVE_EVOLUTION creations, not a dedicated decline-from-X-to-Y group."
3,A-04,"AC-01d, CC-01, CC-02",yes,"Comments independently record missed radio counterparts, radio measurements, and missing radio flux."
4,A-05,AC-01b,yes,Thirty created localization spans include BAT/XRT/UVOT coordinates and confidence radii.
5,A-06,"AC-01e, AC-03, BC-05",partial,"Comments show missed high-energy quantities and modality relabels, but not every Slack instrument-binding case."
6,A-07,BC-05,yes,A structured correction explicitly says event redshift rather than REDSHIFT_CONTEXT.
7,A-08,CC-01,yes,A comment explicitly identifies a colour expression as photometric rather than redshift.
8,A-09,CC-01,yes,A comment explicitly identifies the SN1998bw-like statement as classification information.
9,A-10,AC-02,yes,Five TRIGGER_INSTRUMENT comments explicitly deny trigger role; the comments support and extend the Slack case set.


Symptoms confirmed by BOTH sources:
A-01 <-> AC-01a; A-04 <-> AC-01d, CC-01, CC-02; A-05 <-> AC-01b; A-07 <-> BC-05; A-08 <-> CC-01; A-09 <-> CC-01; A-10 <-> AC-02; A-11 <-> AC-02; A-12 <-> AC-03, DC-03; B-01 <-> BC-01, BC-03, DC-03; B-02 <-> BC-03, DC-03; B-04 <-> BC-05; B-05 <-> BC-01, BC-04, DC-03; B-06 <-> BC-01, BC-02; B-07 <-> BC-01, BC-04; B-08 <-> BC-01, BC-04; C-03 <-> CC-02; D-01 <-> DC-02; D-05 <-> BC-04, DC-03; D-08 <-> DC-01; D-09 <-> DC-01; D-10 <-> CC-01

Symptoms only in Slack:
C-04, C-06, D-03, D-04

Symptoms only in comments:
AC-01c, AC-01f

Partial correspondences (not counted as confirmed by both):
A-02 <-> AC-01a; A-03 <-> AC-01r; A-06 <-> AC-01e, AC-03, BC-05; B-03 <-> BC-03, BC-05; C-01 <-> DC-02; C-02 <-> AC-01e, CC-01; C-05 <-> AC-01r; D-02 <-> BC-05, DC-03; D-06 <-> BC-04, DC-03; D-07 <-> AC-01a; D-11 <-> BC-01, BC-04

DC-01 versus D-08: SUPPORTS AND EXTENDS. DC-01 has 152 comments, 2 annotators, and 4 documents. It directly supports MASTER/amateur/GRANDMA exc

,backlog_id,comments
3,DC-02,40
4,NOT ACTIONABLE,30
0,AC-01c,17
1,AC-02,5
2,BC-05,4



Wrote /home/meneses/project_astronomical/MAFORAI/docs/gcn_gold_corpus/BACKLOG_CROSSWALK.md (69 generated lines)

This notebook remains a qualitative backlog derivation, not a measurement.
